In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

from src import model_utils, text_preprocessing
from config import PROCESSED_DATA_DIR, MODELS_DIR

In [2]:
reviews_df = pd.read_parquet(PROCESSED_DATA_DIR / "amazon_reviews_features.parquet")

In [3]:
reviews_df = model_utils.drop_neutral_ratings(reviews_df)
reviews_df["rating"] = reviews_df["rating"].apply(model_utils.classify_rating)

reviews_df["tfidf_text"] = reviews_df["full_text"].apply(text_preprocessing.clean_text)
reviews_df = model_utils.drop_non_feature_columns(reviews_df.copy())
train_df, val_df, test_df = model_utils.split_data(reviews_df)

y_tr = train_df["rating"]
X_tr = train_df.drop(columns=["rating"])

In [4]:
text_feature = "tfidf_text"
numeric_features = ["log_review_count", "text_length"]
country_features = [col for col in train_df.columns if col.startswith("country_")]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(stop_words="english"),
            text_feature,
        ),
        ("num", StandardScaler(), numeric_features),
        ("country", "passthrough", country_features),
    ]
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

param_grid = {
    "preprocessor__text__max_features": [1000, 5000, 10000],
    "preprocessor__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessor__text__min_df": [3, 5],
    "classifier__C": [0.1, 1, 10],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid_search.fit(X_tr, y_tr)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1: {grid_search.best_score_:.4f}")

Best parameters: {'classifier__C': 10, 'preprocessor__text__max_features': 10000, 'preprocessor__text__min_df': 3, 'preprocessor__text__ngram_range': (1, 2)}
Best CV F1: 0.9135


In [5]:
best_model = grid_search.best_estimator_

model_utils.test_model(best_model, val_df, split_name="Validation")
model_utils.test_model(best_model, test_df, split_name="Test")

Validation accuracy: 0.9496777392166584
Validation macro F1: 0.9390116485307012
Validation weighted F1: 0.9497978670904185

Validation classification report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      2870
           1       0.91      0.92      0.91      1164

    accuracy                           0.95      4034
   macro avg       0.94      0.94      0.94      4034
weighted avg       0.95      0.95      0.95      4034


Validation confusion matrix:
[[2759  111]
 [  92 1072]]
Test accuracy: 0.9489340604858701
Test macro F1: 0.9380636109739722
Test weighted F1: 0.9490369759842076

Test classification report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      2870
           1       0.91      0.92      0.91      1164

    accuracy                           0.95      4034
   macro avg       0.94      0.94      0.94      4034
weighted avg       0.95      0.95      0.95      403

In [6]:
# Save tuned model to disk
saved_path = model_utils.save_model(best_model, MODELS_DIR / "binary_logreg_tuned.joblib")
print(saved_path)

/Users/brandonwu/Documents/amazon_reviews_project/models/binary_logreg_tuned.joblib
